In [64]:
import base64, io
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
import plotly.express as px
from google.colab import drive
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
from collections import defaultdict
from torch.utils.data import Dataset
from matplotlib.patches import Rectangle
from plotly.subplots import make_subplots
import os, cv2, json, torch, random, zipfile
from sklearn.model_selection import GroupKFold

In [2]:
SEED = 42
SUBSAMPLING_STEP = 5
N_FOLDS = 5
FOLD = 0
TARGET_MIN_SIZE = 480
TARGET_MAX_SIZE = 800
HORIZONTAL_FLIP_P = 0.5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print('Seed:', SEED)

Seed: 42


### environment setup

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
zip_path = "/content/drive/MyDrive/multiple-object-video-detection-hw-2-ucu-ai-2026.zip"
extract_path = "/content/datasets"
dataset_path = os.path.join(extract_path, "multiple-object-video-detection-hw-2-ucu-ai-2026")

os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as archive:
    files = [file for file in archive.infolist() if not file.filename.startswith("__MACOSX/") and not file.is_dir()]
    missing_files = [file for file in files if not os.path.isfile(os.path.join(extract_path, file.filename))]

    if not missing_files:
        print("dataset already extracted; skipping extraction")
    else:
        for file in tqdm(missing_files, desc="extracting dataset"):
            archive.extract(file, extract_path)

print("\ndataset path:", dataset_path)
print("dataset contents:", os.listdir(dataset_path))

extracting dataset:   0%|          | 0/77434 [00:00<?, ?it/s]


dataset path: /content/datasets/multiple-object-video-detection-hw-2-ucu-ai-2026
dataset contents: ['test', 'train', 'test_images.csv', 'val']


### dataset overview and class distribution


In [23]:

dataset_path = "/content/datasets/multiple-object-video-detection-hw-2-ucu-ai-2026"
train_path = os.path.join(dataset_path, "train", "annotations_coco.json")
val_path = os.path.join(dataset_path, "val", "annotations_coco.json")

with open(train_path, "r") as file:
    train_data = json.load(file)
with open(val_path, "r") as file:
    val_data = json.load(file)

categories = {category["id"]: category["name"] for category in train_data["categories"]}

train_counts = pd.Series([annotation["category_id"] for annotation in train_data["annotations"] if annotation["iscrowd"] == 0]).value_counts()
val_counts = pd.Series([annotation["category_id"] for annotation in val_data["annotations"] if annotation["iscrowd"] == 0]).value_counts()

class_distribution = pd.DataFrame({"train": train_counts, "validation": val_counts}).fillna(0).astype(int)
class_distribution.index = [categories[category_id] for category_id in class_distribution.index]
class_distribution = class_distribution.sort_values("train", ascending=False)
display(class_distribution.rename_axis(None))

fig = px.bar(class_distribution.reset_index(names="class").melt(id_vars="class", var_name="split", value_name="count"),
             x="class", y="count", color="split", barmode="group", title="Class Distribution",
             color_discrete_map={"train": "#006B55", "validation": "#193378"})
fig.update_layout(xaxis_title="class", yaxis_title="number of annotations", legend_title=None, xaxis_tickangle=-45)
fig.show()


,train,validation
car,505301,31821
person,328701,50312
motor,102819,12025
van,46940,6842
tricycle,41349,5487
bicycle,40255,6022
truck,30498,1359
bus,9653,264


In [7]:
for split_name, data in [('train', train_data), ('official validation', val_data)]:
    print(f'{split_name}: {len(data["images"]):,} frames, '
          f'{len(set(im["video"] for im in data["images"]))} sequences, '
          f'{len(data["annotations"]):,} COCO annotations')
print('target category IDs:', categories)


train: 24,201 frames, 56 sequences, 1,616,260 COCO annotations
official validation: 2,846 frames, 7 sequences, 146,092 COCO annotations
target category IDs: {1: 'person', 2: 'bicycle', 3: 'car', 4: 'van', 5: 'truck', 6: 'tricycle', 7: 'bus', 8: 'motor'}


In [16]:
rows = []

for split_name, data in [("train", train_data), ("validation", val_data)]:
    annotations_df = pd.DataFrame(data["annotations"])
    crowd_by_class = pd.crosstab(annotations_df["category_id"].map(categories), annotations_df["iscrowd"])

    for class_name, counts in crowd_by_class.iterrows():
        rows.append({"class": class_name, "split": split_name, "regular objects": counts.get(0, 0),
            "ignored regions": counts.get(1, 0), "ignored (%)": round(counts.get(1, 0) / counts.sum() * 100, 2)})

crowd_summary = pd.DataFrame(rows)

display(crowd_summary.pivot(index="class", columns="split",
      values=["regular objects", "ignored regions", "ignored (%)"]).round(2).rename_axis(index=None, columns=[None, None]))

regular objects            ignored regions            ignored (%)  \
                   train validation           train validation       train   
bicycle          40255.0     6022.0         63843.0     3995.0       61.33   
bus               9653.0      264.0         63843.0     3995.0       86.87   
car             505301.0    31821.0         63843.0     3995.0       11.22   
motor           102819.0    12025.0         63843.0     3995.0       38.31   
person          328701.0    50312.0         63843.0     3995.0       16.26   
tricycle         41349.0     5487.0         63843.0     3995.0       60.69   
truck            30498.0     1359.0         63843.0     3995.0       67.67   
van              46940.0     6842.0         63843.0     3995.0       57.63   

                     
         validation  
bicycle       39.88  
bus           93.80  
car           11.15  
motor         24.94  
person         7.36  
tricycle      42.13  
truck         74.62  
van           36.86

### ignored regions and non-target categories

In [14]:
rows = []

for split_name, data in [("train", train_data), ("validation", val_data)]:
    annotations_df = pd.DataFrame(data["annotations"])

    crowd_annotations = annotations_df[annotations_df["iscrowd"] == 1].copy()
    crowd_annotations["bbox_key"] = crowd_annotations["bbox"].apply(tuple)

    grouped_regions = crowd_annotations.groupby(["image_id", "bbox_key"]).agg(
        annotations_count=("category_id", "size"), unique_classes=("category_id", "nunique"))

    rows.append({"split": split_name, "crowd annotations": len(crowd_annotations), "unique ignored regions": len(grouped_regions),
        "annotations per region": grouped_regions["annotations_count"].mode().iloc[0], "classes per region": grouped_regions["unique_classes"].mode().iloc[0]})

display(pd.DataFrame(rows).set_index("split").rename_axis(None))

,crowd annotations,unique ignored regions,annotations per region,classes per region
train,510744,63843,8,8
validation,31960,3995,8,8


In [18]:
image_info = train_data["images"][0]
image_id = image_info["id"]
image_width = image_info["width"]
image_height = image_info["height"]

image_relative_path = image_info["file_name"]
ignore_relative_path = image_relative_path.replace("images/", "ignore/").rsplit(".", 1)[0] + ".txt"
ignore_path = os.path.join(dataset_path, "train", ignore_relative_path)

print("image:", image_relative_path)
print("ignore file:", ignore_path)
print("ignore file exists:", os.path.exists(ignore_path))

coco_regions = []

for annotation in train_data["annotations"]:
    if annotation["image_id"] == image_id and annotation["iscrowd"] == 1:
        bbox = annotation["bbox"]

        if bbox not in coco_regions:
            coco_regions.append(bbox)

txt_regions = []

if os.path.exists(ignore_path):
    with open(ignore_path, "r") as file:
        for line in file:
            values = [float(value) for value in line.split()]

            if len(values) == 4:
                x_center, y_center, width, height = values
                x = (x_center - width / 2) * image_width
                y = (y_center - height / 2) * image_height
                w = width * image_width
                h = height * image_height
                txt_regions.append([x, y, w, h])

print("\nCOCO unique ignore regions:", len(coco_regions))
print("TXT ignore regions:", len(txt_regions))
print("\nCOCO regions:")
for bbox in coco_regions[:5]:
    print(bbox)

print("\nTXT regions converted to pixels:")
for bbox in txt_regions[:5]:
    print([round(value, 2) for value in bbox])

matched_regions = 0

for coco_bbox in coco_regions:
    for txt_bbox in txt_regions:
        if np.allclose(coco_bbox, txt_bbox, atol=2):
            matched_regions += 1
            break

print("\nmatched regions:", matched_regions)

image: images/uav0000013_00000_v/0000001.jpg
ignore file: /content/datasets/multiple-object-video-detection-hw-2-ucu-ai-2026/train/ignore/uav0000013_00000_v/0000001.txt
ignore file exists: True

COCO unique ignore regions: 4
TXT ignore regions: 4

COCO regions:
[593, 43, 174, 190]
[489, 307, 41, 62]
[635, 231, 39, 62]
[618, 289, 40, 53]

TXT regions converted to pixels:
[593.0, 43.0, 174.0, 190.0]
[489.0, 307.0, 41.0, 62.0]
[635.0, 231.0, 39.0, 62.0]
[618.0, 289.0, 40.0, 53.0]

matched regions: 4


In [22]:
size_results = []

for split_name, data in [("train", train_data), ("validation", val_data)]:
    for annotation in data["annotations"]:
        if annotation["iscrowd"] == 0:
            width, height = annotation["bbox"][2:4]
            area = width * height
            size_category = "small" if area < 32**2 else "medium" if area < 96**2 else "large"
            size_results.append({"split": split_name, "class": categories[annotation["category_id"]],
                                 "width": width, "height": height, "area": area, "size": size_category})

sizes_df = pd.DataFrame(size_results)
size_distribution = pd.crosstab(sizes_df["split"], sizes_df["size"]).reindex(columns=["small", "medium", "large"], fill_value=0)

display(pd.concat({"count": size_distribution, "percentage (%)": size_distribution.div(size_distribution.sum(axis=1), axis=0).mul(100).round(2)}, axis=1).rename_axis(index=None, columns=[None, None]))

fig = px.bar(size_distribution.reset_index().melt(id_vars="split", var_name="size", value_name="count"),
             x="split", y="count", color="size", barmode="stack", title="Object Size Distribution",
             color_discrete_map={"small": "#006B55", "medium": "#193378", "large": "#9CA3AF"},
             category_orders={"size": ["small", "medium", "large"], "split": ["train", "validation"]})
fig.update_layout(xaxis_title="dataset split", yaxis_title="number of objects", legend_title="object size")
fig.show()


count                 percentage (%)              
             small  medium   large          small medium  large
train       275347  702004  128165          24.91  63.50  11.59
validation   21910   79357   12865          19.20  69.53  11.27

In [25]:
train_sizes = sizes_df[sizes_df["split"] == "train"]

class_size_distribution = pd.crosstab(train_sizes["class"], train_sizes["size"], normalize="index") * 100
class_size_distribution = class_size_distribution.reindex(columns=["small", "medium", "large"], fill_value=0)
class_size_distribution = class_size_distribution.sort_values("small", ascending=False)

display(class_size_distribution.round(2).rename_axis(index=None, columns=None))

fig = px.bar(class_size_distribution.reset_index(names="class").melt(id_vars="class", var_name="size", value_name="percentage"),
             x="class", y="percentage", color="size", barmode="stack", title="Object Size Distribution by Class (train)",
             color_discrete_map={"small": "#006B55", "medium": "#193378", "large": "#9CA3AF"},
             category_orders={"size": ["small", "medium", "large"]})
fig.update_layout(xaxis_title="object class", yaxis_title="percentage of objects (%)", legend_title="object size", xaxis_tickangle=-45)
fig.show()

,small,medium,large
person,39.49,60.17,0.34
motor,30.37,68.22,1.42
bicycle,24.41,74.11,1.48
van,21.40,62.74,15.85
tricycle,21.03,66.13,12.84
car,16.32,64.52,19.15
truck,9.39,57.90,32.71
bus,4.14,38.97,56.88


In [28]:
frame_results = []

for split_name, data in [("train", train_data), ("validation", val_data)]:
    annotations_df = pd.DataFrame(data["annotations"])
    annotations_df = annotations_df[annotations_df["iscrowd"] == 0]
    object_counts = annotations_df.groupby("image_id").size()

    for image in data["images"]:
        frame_results.append({"split": split_name, "image id": image["id"], "video": image["video"],
                              "object count": int(object_counts.get(image["id"], 0))})

frames_df = pd.DataFrame(frame_results)
frame_statistics = frames_df.groupby("split")["object count"].agg(["count", "mean", "median", "std", "min", "max"])
display(frame_statistics.round(2).rename_axis(None))

fig = px.histogram(frames_df, x="object count", facet_col="split", nbins=50,
                   color="split", title="Objects per Frame",
                   color_discrete_map={"train": "#006B55", "validation": "#193378"},
                   category_orders={"split": ["train", "validation"]})
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(showlegend=False, bargap=0.03)
fig.update_xaxes(title_text="number of objects", title_standoff=10)
fig.update_yaxes(title_text="number of frames", title_standoff=5)
fig.show()

,count,mean,median,std,min,max
train,24201,45.68,43.0,31.56,0,180
validation,2846,40.10,37.0,26.85,8,116


In [29]:
video_statistics = frames_df.groupby(["split", "video"]).agg(
    frames=("image id", "count"), mean_objects=("object count", "mean"),
    median_objects=("object count", "median"), max_objects=("object count", "max"),
    total_objects=("object count", "sum")).reset_index()

display(video_statistics.head(10).round(2).rename(columns=lambda x: x.replace("_", " ")).style.hide(axis="index"))

video_summary = video_statistics.groupby("split")[["frames", "mean_objects", "max_objects"]].agg(["min", "median", "max"]).round(2)
display(video_summary.rename_axis(index=None, columns=[None, None]).rename(columns=lambda x: x.replace("_", " "), level=0))

fig = px.bar(video_statistics.sort_values("mean_objects", ascending=False), x="video", y="mean_objects",
             color="split", facet_col="split", title="Average Objects per Frame by Video",
             color_discrete_map={"train": "#006B55", "validation": "#193378"},
             category_orders={"split": ["train", "validation"]})
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(showlegend=False, xaxis_title="video sequences", yaxis_title="average number of objects")
fig.update_xaxes(showticklabels=False, matches=None, title_text="video sequences", title_standoff=10)
fig.update_yaxes(title_text="average number of objects", title_standoff=5)
fig.show()

split,video,frames,mean objects,median objects,max objects,total objects
train,uav0000013_00000_v,269,28.950000,30.000000,37,7787
train,uav0000013_01073_v,58,17.740000,18.000000,23,1029
train,uav0000013_01392_v,118,23.810000,24.000000,29,2810
train,uav0000020_00406_v,501,18.680000,19.000000,23,9359
train,uav0000071_03240_v,181,29.960000,26.000000,44,5422
train,uav0000072_04488_v,85,110.380000,108.000000,124,9382
train,uav0000072_05448_v,217,47.690000,39.000000,81,10348
train,uav0000072_06432_v,97,89.950000,85.000000,113,8725
train,uav0000076_00720_v,361,19.710000,21.000000,30,7114
train,uav0000079_00480_v,361,34.200000,30.000000,59,12347


frames              mean objects                max objects         \
              min median   max          min median     max         min median   
train          58  383.5  1424         5.15  43.76  142.96           7   59.5   
validation    184  349.0   978        13.88  48.68  103.38          21   71.0   

                 
            max  
train       180  
validation  116

In [30]:
seq_rows = []
for split_name, data in [("train", train_data), ("validation", val_data)]:
    im_to_video = {im["id"]: im["video"] for im in data["images"]}

    for ann in data["annotations"]:
        video = im_to_video.get(ann["image_id"])
        if video is None:
            continue

        w, h = ann["bbox"][2:4]
        seq_rows.append({"split": split_name, "video": video, "class": categories.get(ann["category_id"], "non-target"),
                         "ignored": int(ann.get("iscrowd", 0) == 1), "small": int(w * h < 32**2)})

seq_df = pd.DataFrame(seq_rows)
regular_df = seq_df[seq_df["ignored"] == 0]

sequence_mix = regular_df.groupby(["split", "video"]).agg(
    objects=("class", "size"), small_pct=("small", lambda s: 100 * s.mean())).reset_index()

sequence_mix = sequence_mix.merge(video_statistics[["split", "video", "frames", "mean_objects"]],
                                  on=["split", "video"], how="left")

sequence_mix = sequence_mix.rename(columns={"small_pct": "small (%)", "mean_objects": "mean objects"})
display(sequence_mix.sort_values(["split", "small (%)"], ascending=[True, False]).round(2).style.hide(axis="index"))

class_by_sequence = pd.crosstab([regular_df["split"], regular_df["video"]], regular_df["class"], normalize="index") * 100
display(class_by_sequence.round(1).rename_axis(index=[None, None], columns=None))

split,video,objects,small (%),frames,mean objects
train,uav0000248_00001_v,32154,73.090000,677,47.490000
train,uav0000239_12336_v,11342,65.180000,341,33.260000
train,uav0000264_02760_v,11749,54.470000,616,19.070000
train,uav0000278_00001_v,52919,51.070000,962,55.010000
train,uav0000239_03720_v,15653,47.830000,680,23.020000
train,uav0000279_00001_v,52155,47.570000,547,95.350000
train,uav0000145_00000_v,3450,46.200000,307,11.240000
train,uav0000243_00001_v,17620,44.370000,768,22.940000
train,uav0000084_00000_v,27907,39.070000,516,54.080000
train,uav0000244_01440_v,40928,37.600000,721,56.770000


bicycle  bus   car  motor  person  tricycle  \
train      uav0000013_00000_v      1.6  0.0   0.0   21.4    74.2       2.8   
           uav0000013_01073_v      0.0  0.0   0.0   17.3    82.7       0.0   
           uav0000013_01392_v      0.0  0.0   0.0   26.3    67.6       6.1   
           uav0000020_00406_v      3.3  0.0   0.0   13.4    79.3       3.9   
           uav0000071_03240_v      4.7  0.0  33.7   10.2    48.1       2.1   
...                                ...  ...   ...    ...     ...       ...   
validation uav0000137_00458_v     19.8  0.0  27.3   12.3    38.6       0.0   
           uav0000182_00000_v      0.8  0.0  33.1   33.3     6.6      13.0   
           uav0000268_05773_v      0.0  0.0  64.0    0.0    14.6       3.8   
           uav0000305_00000_v      0.0  4.1  56.6   16.7     9.4       3.5   
           uav0000339_00001_v      6.3  0.0  24.7    5.3    45.6       9.8   

                               truck   van  
train      uav0000013_00000_v    0.0   0.0  
           uav0000013_01073_v    0.0   0.0  
           uav0000013_01392_v    0.0   0.0  
           uav0000020_00406_v    0.0   0.0  
           uav0000071_03240_v    0.0   1.2  
...                              ...   ...  
validation uav0000137_00458_v    0.0   2.0  
           uav0000182_00000_v    2.0  11.1  
           uav0000268_05773_v    2.4  15.2  
           uav0000305_00000_v    1.2   8.5  
           uav0000339_00001_v    1.1   7.1  

[63 rows x 8 columns]

### subsampling and leakage-free sequence split

We keep every fifth frame from each video sequence based on sorted frame_id, because nearby frames are very similar. This helps reduce the amount of data and speed up training.

We use GroupKFold to split the data so that frames from the same video do not appear in both the training and validation sets.

In [58]:
train_images_df = pd.DataFrame(train_data["images"])
train_images_df = train_images_df.sort_values(["video", "frame_id"]).reset_index(drop=True)
train_images_df["frame position"] = train_images_df.groupby("video").cumcount()

subsampled_images_df = train_images_df[train_images_df["frame position"] % SUBSAMPLING_STEP == 0].copy()

subsampling_summary = pd.DataFrame({"metric": ["frames", "videos"], "original": [len(train_images_df), train_images_df["video"].nunique()],
    "subsampled": [len(subsampled_images_df), subsampled_images_df["video"].nunique()]})

subsampling_summary["retained (%)"] = (subsampling_summary["subsampled"] / subsampling_summary["original"] * 100).round(2)

display(subsampling_summary.style.hide(axis="index"))

metric,original,subsampled,retained (%)
frames,24201,4868,20.110000
videos,56,56,100.000000


In [38]:
selected_image_ids = set(subsampled_images_df["id"])

train_annotations_df = pd.DataFrame(train_data["annotations"])
train_annotations_df = train_annotations_df[train_annotations_df["iscrowd"] == 0].copy()
sampled_annotations_df = train_annotations_df[train_annotations_df["image_id"].isin(selected_image_ids)].copy()

full_class_counts = train_annotations_df["category_id"].value_counts(normalize=True).sort_index() * 100
sampled_class_counts = sampled_annotations_df["category_id"].value_counts(normalize=True).sort_index() * 100

class_comparison = pd.DataFrame({"full train (%)": full_class_counts, "subsampled (%)": sampled_class_counts}).fillna(0)
class_comparison["difference (pp)"] = class_comparison["subsampled (%)"] - class_comparison["full train (%)"]
class_comparison.index = [categories[category_id] for category_id in class_comparison.index]

print("\nclass distribution comparison:")
display(class_comparison.round(2).rename_axis(None))

sampled_annotations_df["size"] = sampled_annotations_df["bbox"].apply(
    lambda bbox: "small" if bbox[2] * bbox[3] < 32**2 else "medium" if bbox[2] * bbox[3] < 96**2 else "large")

full_size_counts = sizes_df[sizes_df["split"] == "train"]["size"].value_counts(normalize=True) * 100
sampled_size_counts = sampled_annotations_df["size"].value_counts(normalize=True) * 100

size_comparison = pd.DataFrame({"full train (%)": full_size_counts, "subsampled (%)": sampled_size_counts}).reindex(["small", "medium", "large"]).fillna(0)
size_comparison["difference (pp)"] = size_comparison["subsampled (%)"] - size_comparison["full train (%)"]

print("\nobject size comparison:")
display(size_comparison.round(2).rename_axis(None))

full_mean = frames_df[frames_df["split"] == "train"]["object count"].mean()
sampled_mean = frames_df[(frames_df["split"] == "train") & (frames_df["image id"].isin(selected_image_ids))]["object count"].mean()

mean_comparison = pd.DataFrame({"dataset": ["full train", "subsampled"], "mean objects per frame": [full_mean, sampled_mean]})

print("\nobjects per frame comparison:")
display(mean_comparison.round(2).style.hide(axis="index"))



class distribution comparison:


,full train (%),subsampled (%),difference (pp)
person,29.73,29.77,0.04
bicycle,3.64,3.65,0.01
car,45.71,45.68,-0.03
van,4.25,4.24,-0.01
truck,2.76,2.75,-0.01
tricycle,3.74,3.74,0.00
bus,0.87,0.87,-0.00
motor,9.30,9.29,-0.01



object size comparison:


,full train (%),subsampled (%),difference (pp)
small,24.91,24.96,0.05
medium,63.50,63.46,-0.04
large,11.59,11.58,-0.02



objects per frame comparison:


dataset,mean objects per frame
full train,45.680000
subsampled,45.780000


In [37]:
fig = px.bar(class_comparison[["full train (%)", "subsampled (%)"]].reset_index(names="class").melt(id_vars="class", var_name="dataset", value_name="percentage"),
             x="class", y="percentage", color="dataset", barmode="group", title="Class Distribution Comparison",
             color_discrete_map={"full train (%)": "#006B55", "subsampled (%)": "#193378"})
fig.update_layout(xaxis_title="class", yaxis_title="percentage (%)", legend_title=None, xaxis_tickangle=-45)
fig.update_xaxes(title_standoff=10)
fig.update_yaxes(title_standoff=5)
fig.show()

fig = px.bar(size_comparison[["full train (%)", "subsampled (%)"]].reset_index(names="size").melt(id_vars="size", var_name="dataset", value_name="percentage"),
             x="size", y="percentage", color="dataset", barmode="group", title="Object Size Comparison",
             color_discrete_map={"full train (%)": "#006B55", "subsampled (%)": "#193378"},
             category_orders={"size": ["small", "medium", "large"]})
fig.update_layout(xaxis_title="object size", yaxis_title="percentage (%)", legend_title=None)
fig.update_xaxes(title_standoff=10)
fig.update_yaxes(title_standoff=5)
fig.show()

In [59]:
cv_images_df = subsampled_images_df.copy()
cv_images_df["fold"] = -1
group_kfold = GroupKFold(n_splits=N_FOLDS)
fold_rows = []

for fold, (train_indices, val_indices) in enumerate(group_kfold.split(cv_images_df, groups=cv_images_df["video"])):
    cv_images_df.iloc[val_indices, cv_images_df.columns.get_loc("fold")] = fold

    train_videos = set(cv_images_df.iloc[train_indices]["video"])
    val_videos = set(cv_images_df.iloc[val_indices]["video"])
    overlapping_videos = len(train_videos & val_videos)

    assert train_videos.isdisjoint(val_videos)

    fold_rows.append({"fold": fold + 1, "train frames": len(train_indices), "validation frames": len(val_indices),
                      "train videos": len(train_videos), "validation videos": len(val_videos),
                      "overlapping videos": overlapping_videos})

assert (cv_images_df["fold"] >= 0).all()

print("\ncross-validation fold distribution:")
display(pd.DataFrame(fold_rows).style.hide(axis="index"))

fold_summary = cv_images_df.groupby("fold").agg(frames=("id", "count"), videos=("video", "nunique"))

print("\nvalidation frames and videos per fold:")
display(fold_summary.rename_axis(None))

train_video_names = set(cv_images_df["video"])
official_val_video_names = {image["video"] for image in val_data["images"]}
official_overlap = len(train_video_names & official_val_video_names)

official_summary = pd.DataFrame({
    "frames": [len(val_data["images"])],
    "videos": [len(official_val_video_names)],
    "overlapping videos": [official_overlap]
})

print("\nofficial validation summary:")
display(official_summary.style.hide(axis="index"))

assert train_video_names.isdisjoint(official_val_video_names)

output_dir = "/content/drive/MyDrive/ai_hw2_results"
os.makedirs(output_dir, exist_ok=True)

folds_path = os.path.join(output_dir, "sequence_folds.csv")
cv_images_df[["id", "video", "frame_id", "file_name", "fold"]].to_csv(folds_path, index=False)

print(f"\nsaved folds to: {folds_path}")


cross-validation fold distribution:


fold,train frames,validation frames,train videos,validation videos,overlapping videos
1,3887,981,45,11,0
2,3889,979,45,11,0
3,3905,963,45,11,0
4,3894,974,44,12,0
5,3897,971,45,11,0



validation frames and videos per fold:


,frames,videos
0,981,11
1,979,11
2,963,11
3,974,12
4,971,11



official validation summary:


frames,videos,overlapping videos
2846,7,0



saved folds to: /content/drive/MyDrive/ai_hw2_results/sequence_folds.csv


In [60]:
distribution_check = pd.DataFrame({
    "metric": ["class distribution difference (pp)", "object size difference (pp)"],
    "max absolute difference": [class_comparison["difference (pp)"].abs().max(), size_comparison["difference (pp)"].abs().max()]})

print("subsampling distribution comparison:")
display(distribution_check.round(2).style.hide(axis="index"))

assert cv_images_df.groupby("video")["fold"].nunique().eq(1).all()
assert cv_images_df["id"].is_unique
assert cv_images_df["fold"].between(0, N_FOLDS - 1).all()
print("\nsequence leakage check: passed")

subsampling distribution comparison:


metric,max absolute difference
class distribution difference (pp),0.040000
object size difference (pp),0.050000



sequence leakage check: passed


### COCO annotation conversion for PyTorch detection


In [43]:
target_category_ids = sorted(categories)
category_to_label = {category_id: i + 1 for i, category_id in enumerate(target_category_ids)}
label_to_category = {label: category_id for category_id, label in category_to_label.items()}

print("coco category to pytorch label:")
display(pd.DataFrame([{"category id": category_id, "class": categories[category_id], "pytorch label": label}
                      for category_id, label in category_to_label.items()]).style.hide(axis="index"))

coco category to pytorch label:


category id,class,pytorch label
1,person,1
2,bicycle,2
3,car,3
4,van,4
5,truck,5
6,tricycle,6
7,bus,7
8,motor,8


In [44]:
class VisDroneDataset(Dataset):
    def __init__(self, dataset_path, split, coco_data, image_ids=None, augment=False):
        self.dataset_path = dataset_path
        self.split = split
        self.augment = augment
        self.images = coco_data["images"]

        if image_ids is not None:
            selected_ids = set(image_ids)
            self.images = [image for image in self.images if image["id"] in selected_ids]

        self.annotations = defaultdict(list)
        self.ignore_annotations = defaultdict(set)

        for annotation in coco_data["annotations"]:
            image_id = annotation["image_id"]

            if annotation.get("iscrowd", 0) == 0 and annotation["category_id"] in category_to_label:
                self.annotations[image_id].append(annotation)
            else:
                self.ignore_annotations[image_id].add(tuple(annotation["bbox"]))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image_info = self.images[index]
        image_path = os.path.join(self.dataset_path, self.split, image_info["file_name"])

        image = Image.open(image_path).convert("RGB")
        image_width, image_height = image.size
        boxes, labels, ignored_boxes = [], [], []

        for annotation in self.annotations[image_info["id"]]:
            x, y, width, height = annotation["bbox"]
            xmin, ymin = max(0, x), max(0, y)
            xmax, ymax = min(image_width, x + width), min(image_height, y + height)

            if xmax <= xmin or ymax <= ymin:
                continue

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(category_to_label[annotation["category_id"]])

        for x, y, width, height in self.ignore_annotations[image_info["id"]]:
            xmin, ymin = max(0, x), max(0, y)
            xmax, ymax = min(image_width, x + width), min(image_height, y + height)

            if xmax > xmin and ymax > ymin:
                ignored_boxes.append([xmin, ymin, xmax, ymax])

        boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.tensor(labels, dtype=torch.int64)
        ignored_boxes = torch.tensor(ignored_boxes, dtype=torch.float32).reshape(-1, 4)
        image = torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255.0

        if self.augment and torch.rand(1).item() < 0.5:
            image = torch.flip(image, dims=[2])

            for box_set in [boxes, ignored_boxes]:
                if len(box_set) > 0:
                    old_xmin, old_xmax = box_set[:, 0].clone(), box_set[:, 2].clone()
                    box_set[:, 0] = image_width - old_xmax
                    box_set[:, 2] = image_width - old_xmin

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor(image_info["id"]),
            "ignored_boxes": ignored_boxes
        }

        return image, target

In [45]:
train_dataset = VisDroneDataset(dataset_path=dataset_path, split="train", coco_data=train_data, image_ids=subsampled_images_df["id"], augment=True)
analysis_dataset = VisDroneDataset(dataset_path=dataset_path, split="train", coco_data=train_data, image_ids=subsampled_images_df["id"], augment=False)
val_dataset = VisDroneDataset(dataset_path=dataset_path, split="val", coco_data=val_data, augment=False)

train_image, train_target = analysis_dataset[0]
val_image, val_target = val_dataset[0]

dataset_summary = pd.DataFrame({"dataset": ["train", "validation"], "images": [len(train_dataset), len(val_dataset)],
    "image shape": [tuple(train_image.shape), tuple(val_image.shape)], "boxes": [len(train_target["boxes"]), len(val_target["boxes"])],
    "ignored regions": [len(train_target["ignored_boxes"]), len(val_target["ignored_boxes"])]})

print("dataset summary (boxes and ignored regions for the first image):")
display(dataset_summary.style.hide(axis="index"))

assert train_image.shape[0] == 3
assert len(train_target["boxes"]) == len(train_target["labels"])
assert train_target["boxes"].ndim == 2 and train_target["boxes"].shape[1] == 4
assert train_target["ignored_boxes"].ndim == 2 and train_target["ignored_boxes"].shape[1] == 4
assert all(int(label) in category_to_label.values() for label in train_target["labels"])
assert train_dataset.augment is True
assert analysis_dataset.augment is False
assert val_dataset.augment is False

print("\ndataset checks passed!")

dataset summary (boxes and ignored regions for the first image):


dataset,images,image shape,boxes,ignored regions
train,4868,"(3, 756, 1344)",23,4
validation,2846,"(3, 756, 1344)",36,0



dataset checks passed!


In [46]:
fold = 0

fold_train_ids = cv_images_df[cv_images_df["fold"] != fold]["id"]
fold_val_ids = cv_images_df[cv_images_df["fold"] == fold]["id"]

fold_train_dataset = VisDroneDataset(dataset_path=dataset_path, split="train", coco_data=train_data, image_ids=fold_train_ids, augment=True)
fold_val_dataset = VisDroneDataset(dataset_path=dataset_path, split="train", coco_data=train_data, image_ids=fold_val_ids, augment=False)

train_video_names = {image["video"] for image in fold_train_dataset.images}
val_video_names = {image["video"] for image in fold_val_dataset.images}

fold_summary = pd.DataFrame({"fold": [fold + 1], "train frames": [len(fold_train_dataset)], "validation frames": [len(fold_val_dataset)],
    "train videos": [len(train_video_names)], "validation videos": [len(val_video_names)], "overlapping videos": [len(train_video_names & val_video_names)]})

print("cross-validation dataset summary:")
display(fold_summary.style.hide(axis="index"))
assert train_video_names.isdisjoint(val_video_names)
assert len(fold_train_dataset) + len(fold_val_dataset) == len(subsampled_images_df)
print("\ncross-validation dataset checks passed!")

cross-validation dataset summary:


fold,train frames,validation frames,train videos,validation videos,overlapping videos
1,3887,981,45,11,0



cross-validation dataset checks passed!


In [65]:
by_video = {}
for idx, info in enumerate(analysis_dataset.images):
    by_video.setdefault(info["video"], idx)

samples = [("train", analysis_dataset, idx) for idx in list(by_video.values())[:3]]
if len(val_dataset):
    samples.append(("official validation", val_dataset, 0))

titles = [f'{split_name}: {ds.images[idx]["video"]} | {len(ds[idx][1]["boxes"])} targets, {len(ds[idx][1]["ignored_boxes"])} ignored'
          for split_name, ds, idx in samples]

fig = make_subplots(rows=2, cols=2, subplot_titles=titles, horizontal_spacing=0.025, vertical_spacing=0.09)

for i, (split_name, ds, idx) in enumerate(samples):
    row, col = divmod(i, 2)
    row, col = row + 1, col + 1
    image, target = ds[idx]

    image_np = (image.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    buffer = io.BytesIO()
    Image.fromarray(image_np).save(buffer, format="PNG", optimize=True)
    image_source = f"data:image/png;base64,{base64.b64encode(buffer.getvalue()).decode('utf-8')}"

    fig.add_trace(go.Image(source=image_source), row=row, col=col)

    for box, label in zip(target["boxes"].tolist(), target["labels"].tolist()):
        x1, y1, x2, y2 = box
        fig.add_shape(type="rect", x0=x1, y0=y1, x1=x2, y1=y2,
                      line=dict(color="#006B55", width=1), row=row, col=col)
        fig.add_annotation(x=x1, y=y1, text=categories[label_to_category[int(label)]],
                           showarrow=False, xanchor="left", yanchor="bottom",
                           font=dict(size=9, color="white"), bgcolor="#006B55", row=row, col=col)

    for x1, y1, x2, y2 in target["ignored_boxes"].tolist():
        fig.add_shape(type="rect", x0=x1, y0=y1, x1=x2, y1=y2,
                      line=dict(color="red", width=1.5, dash="dash"), row=row, col=col)

fig.update_xaxes(visible=False, constrain="domain")
fig.update_yaxes(visible=False, scaleanchor=None, constrain="domain")
fig.update_layout(title="Ground-Truth Boxes After Annotation Conversion", width=1250, height=760,
                  showlegend=False, margin=dict(l=5, r=5, t=75, b=5),
                  paper_bgcolor="white", plot_bgcolor="white")
fig.show()

In [51]:
issues = []

for i, info in enumerate(analysis_dataset.images):
    _, target = analysis_dataset[i]
    boxes, labels = target["boxes"], target["labels"]

    if not (boxes.ndim == 2 and boxes.shape[1] == 4 and len(boxes) == len(labels)):
        issues.append((info["id"], "shape"))
        continue

    if len(boxes) and (not bool(((boxes[:, 2:] - boxes[:, :2]) > 0).all()) or not bool((boxes[:, 0] >= 0).all() and (boxes[:, 1] >= 0).all())
        or not bool((boxes[:, 2] <= info["width"]).all() and (boxes[:, 3] <= info["height"]).all())):
        issues.append((info["id"], "coordinates"))

    if len(labels) and not set(labels.tolist()).issubset(set(category_to_label.values())):
        issues.append((info["id"], "labels"))

conversion_summary = pd.DataFrame({"frames checked": [len(analysis_dataset)], "issues": [len(issues)], "status": ["passed" if not issues else "failed"]})

print("annotation conversion consistency:")
display(conversion_summary.style.hide(axis="index"))
assert not issues, issues[:10]

annotation conversion consistency:


frames checked,issues,status
4868,0,passed


### image preprocessing, resizing, normalization and augmentation


In [52]:
image, target = analysis_dataset[0]

preprocessing_summary = pd.DataFrame({"metric": ["augmentation", "image dtype", "image shape", "pixel minimum", "pixel maximum",
    "object boxes", "labels", "ignored boxes"], "value": [str(analysis_dataset.augment), str(image.dtype), str(tuple(image.shape)),
    round(image.min().item(), 4), round(image.max().item(), 4), str(tuple(target["boxes"].shape)), str(tuple(target["labels"].shape)),
    str(tuple(target["ignored_boxes"].shape))]})

augmentation_summary = pd.DataFrame({"dataset": ["train", "analysis", "validation"],
    "augmentation": [train_dataset.augment, analysis_dataset.augment, val_dataset.augment]})

print("augmentation configuration:")
display(augmentation_summary.style.hide(axis="index"))

print("\npreprocessing configuration (first analysis image):")
display(preprocessing_summary.style.hide(axis="index"))

assert image.dtype == torch.float32
assert image.shape[0] == 3
assert image.min() >= 0
assert image.max() <= 1
assert len(target["boxes"]) == len(target["labels"])

print("\npreprocessing configuration checks passed!")

augmentation configuration:


dataset,augmentation
train,True
analysis,False
validation,False



preprocessing configuration (first analysis image):


metric,value
augmentation,False
image dtype,torch.float32
image shape,"(3, 756, 1344)"
pixel minimum,0.000000
pixel maximum,1.000000
object boxes,"(23, 4)"
labels,"(23,)"
ignored boxes,"(4, 4)"



preprocessing configuration checks passed!


In [66]:
original_image, original_target = analysis_dataset[0]
flipped_image = torch.flip(original_image, dims=[2])
image_width = original_image.shape[2]

flipped_boxes = original_target["boxes"].clone()
flipped_ignored = original_target["ignored_boxes"].clone()

for boxes in [flipped_boxes, flipped_ignored]:
    if len(boxes) > 0:
        old_xmin, old_xmax = boxes[:, 0].clone(), boxes[:, 2].clone()
        boxes[:, 0] = image_width - old_xmax
        boxes[:, 2] = image_width - old_xmin

fig = make_subplots(rows=1, cols=2, subplot_titles=["Original Image", "Horizontally Flipped Image"],
                    horizontal_spacing=0.025)

for col, (image, boxes, ignored) in enumerate([
    (original_image, original_target["boxes"], original_target["ignored_boxes"]),
    (flipped_image, flipped_boxes, flipped_ignored)
], start=1):
    image_np = (image.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    buffer = io.BytesIO()
    Image.fromarray(image_np).save(buffer, format="PNG", optimize=True)
    image_source = f"data:image/png;base64,{base64.b64encode(buffer.getvalue()).decode('utf-8')}"

    fig.add_trace(go.Image(source=image_source), row=1, col=col)

    for xmin, ymin, xmax, ymax in boxes.tolist():
        fig.add_shape(type="rect", x0=xmin, y0=ymin, x1=xmax, y1=ymax,
                      line=dict(color="#006B55", width=1), row=1, col=col)

    for xmin, ymin, xmax, ymax in ignored.tolist():
        fig.add_shape(type="rect", x0=xmin, y0=ymin, x1=xmax, y1=ymax,
                      line=dict(color="red", width=2, dash="dash"), row=1, col=col)

fig.update_xaxes(visible=False, constrain="domain")
fig.update_yaxes(visible=False, scaleanchor=None, constrain="domain")
fig.update_layout(title="Horizontal Flip Augmentation", width=1250, height=420,
                  margin=dict(l=5, r=5, t=65, b=5), showlegend=False)
fig.show()

assert torch.allclose(flipped_boxes[:, 0], image_width - original_target["boxes"][:, 2])
assert torch.allclose(flipped_ignored[:, 0], image_width - original_target["ignored_boxes"][:, 2])

print("horizontal flip checks passed!")

horizontal flip checks passed!


In [54]:
filtering_results = []

for split_name, data in [("train", train_data), ("official validation", val_data)]:
    annotations = data["annotations"]
    target = sum(a.get("iscrowd", 0) == 0 and a["category_id"] in category_to_label for a in annotations)
    ignored = sum(a.get("iscrowd", 0) != 0 for a in annotations)
    non_target = sum(a.get("iscrowd", 0) == 0 and a["category_id"] not in category_to_label for a in annotations)
    filtering_results.append({"dataset": split_name, "positive annotations": target, "crowd annotations": ignored, "non-target annotations": non_target})

print("annotation filtering summary:")
display(pd.DataFrame(filtering_results).style.hide(axis="index").format({"positive annotations": "{:,}", "crowd annotations": "{:,}", "non-target annotations": "{:,}"}))
print("\nfiltering policy: crowd and non-target boxes are stored separately, not assigned positive labels.")

annotation filtering summary:


dataset,positive annotations,crowd annotations,non-target annotations
train,"1,105,516","510,744",0
official validation,"114,132","31,960",0



filtering policy: crowd and non-target boxes are stored separately, not assigned positive labels.


In [55]:
res_rows = []

for split_name, data in [("train", train_data), ("validation", val_data)]:
    for image in data["images"]:
        res_rows.append({"split": split_name, "width": image["width"], "height": image["height"]})

resolution_df = pd.DataFrame(res_rows)
resolution_summary = resolution_df.groupby(["split", "width", "height"]).size().reset_index(name="frames")

print("image resolution distribution:")
display(resolution_summary.style.hide(axis="index"))

ratio_rows = []

for annotation in train_data["annotations"]:
    width, height = annotation["bbox"][2:4]

    if annotation.get("iscrowd", 0) == 0 and annotation["category_id"] in category_to_label and width > 0 and height > 0:
        ratio_rows.append(width / height)

ratios = pd.Series(ratio_rows, name="width / height")
ratio_summary = ratios.quantile([0.05, 0.25, 0.50, 0.75, 0.95]).reset_index()
ratio_summary.columns = ["quantile", "aspect ratio"]
ratio_summary["quantile"] = (ratio_summary["quantile"] * 100).astype(int).astype(str) + "%"

print("\ntarget box aspect ratio quantiles:")
display(ratio_summary.round(2).style.hide(axis="index"))

print("\nthese statistics describe object shapes but do not define an optimized anchor configuration.")

image resolution distribution:


split,width,height,frames
train,1344,756,5645
train,1360,765,962
train,1904,1071,12895
train,1920,1080,721
train,2688,1512,3505
train,2720,1530,473
validation,1344,756,827
validation,1904,1071,459
validation,2688,1512,233
validation,2720,1530,349



target box aspect ratio quantiles:


quantile,aspect ratio
5%,0.390000
25%,0.590000
50%,0.950000
75%,1.460000
95%,2.450000



these statistics describe object shapes but do not define an optimized anchor configuration.


### brief sum up


In [57]:

train_sizes = sizes_df[sizes_df["split"] == "train"]
small_share = 100 * (train_sizes["size"] == "small").mean()
train_frame_mean = frames_df.loc[frames_df["split"] == "train", "object count"].mean()
seq_train = sequence_mix[sequence_mix["split"] == "train"]

analysis_summary = {"small objects (%)": small_share, "mean objects per frame": train_frame_mean,
    "max class distribution shift (pp)": class_comparison["difference (pp)"].abs().max(),
    "max size distribution shift (pp)": size_comparison["difference (pp)"].abs().max()}

if len(seq_train):
    analysis_summary.update({"min sequence small objects (%)": seq_train["small (%)"].min(),
        "max sequence small objects (%)": seq_train["small (%)"].max(), "min sequence mean objects": seq_train["mean objects"].min(),
        "max sequence mean objects": seq_train["mean objects"].max()})

print("data analysis summary:")
display(pd.DataFrame(analysis_summary.items(), columns=["metric", "value"]).round(2).style.hide(axis="index"))

data analysis summary:


metric,value
small objects (%),24.910000
mean objects per frame,45.680000
max class distribution shift (pp),0.040000
max size distribution shift (pp),0.050000
min sequence small objects (%),0.000000
max sequence small objects (%),73.090000
min sequence mean objects,5.150000
max sequence mean objects,142.960000
